# 🦁 Conser-vision — Wildlife Classification · Google Colab Training

**Competition:** [DrivenData #87](https://www.drivendata.org/competitions/87/)  
**Task:** Multi-class image classification · 8 wildlife classes  
**Metric:** Log-loss (lower is better)

---
## Before you start
1. `Runtime → Change runtime type → T4 GPU`
2. Make sure Google Drive contains the CSV files under `MyDrive/ML_projects/Conser_vision/data/raw/`:
   ```
   MyDrive/ML_projects/Conser_vision/data/raw/
   ├── train_features.csv
   ├── train_labels.csv
   ├── test_features.csv
   └── train_features.zip   ← zipped image folder
   ```
3. Run all cells top-to-bottom with `Runtime → Run all`


## 1 · Environment Setup


In [1]:
# ── GPU check ──────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to: Runtime → Change runtime type → Hardware accelerator → T4 GPU"
    )

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB")
print(f"  torch {torch.__version__}  |  CUDA {torch.version.cuda}")


✓ GPU: Tesla T4  |  VRAM: 15.6 GB
  torch 2.10.0+cu128  |  CUDA 12.8


In [2]:
# ── Mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIGURE THESE PATHS — edit only here                     ║
# ╚══════════════════════════════════════════════════════════════╝
DRIVE_BASE = Path("/content/drive/MyDrive/ML_projects/Conser_vision")
DRIVE_DATA = DRIVE_BASE / "data/raw"
DRIVE_CKPT = DRIVE_BASE / "checkpoints"
DRIVE_SUB  = DRIVE_BASE / "submissions"
# ╚══════════════════════════════════════════════════════════════╝

PROJECT_ROOT = Path("/content/conser-vision")
LOCAL_DATA   = Path("/content/data_local")

for d in [DRIVE_DATA, DRIVE_CKPT, DRIVE_SUB]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive base  : {DRIVE_BASE}")
print(f"Drive data  : {DRIVE_DATA}")
print(f"Drive ckpt  : {DRIVE_CKPT}")


Mounted at /content/drive
Drive base  : /content/drive/MyDrive/ML_projects/Conser_vision
Drive data  : /content/drive/MyDrive/ML_projects/Conser_vision/data/raw
Drive ckpt  : /content/drive/MyDrive/ML_projects/Conser_vision/checkpoints


In [3]:
# ── Install packages ───────────────────────────────────────────
# Colab already has torch, torchvision, numpy, pandas, sklearn, PIL
!pip install -q timm==1.0.3 albumentations==1.4.7 pyyaml tqdm
print("✓ Packages installed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.7/155.7 kB 21.0 MB/s eta 0:00:00
✓ Packages installed


## 2 · Project Source Files *(run once — collapse after)*


In [4]:
# ── Create directory structure ─────────────────────────────────
from pathlib import Path

PROJECT_ROOT = Path("/content/conser-vision")

for d in [
    "src/data", "src/models", "src/training", "src/evaluation",
    "utils", "configs", "data/raw", "data/processed",
    "models/weights", "submissions",
]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

for pkg in ["src", "src/data", "src/models", "src/training", "src/evaluation", "utils"]:
    init = PROJECT_ROOT / pkg / "__init__.py"
    if not init.exists():
        init.write_text("")

print("✓ Directory structure created at", PROJECT_ROOT)


✓ Directory structure created at /content/conser-vision


In [5]:
%%writefile /content/conser-vision/utils/seed.py
"""
utils/seed.py
-------------
Global seed setter for reproducibility.
"""
from __future__ import annotations
import os
import random
import numpy as np


def set_global_seed(seed: int = 42) -> None:
    """Set random seeds for Python, NumPy, PyTorch (and TF if available)."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass
    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
    except ImportError:
        pass


Writing /content/conser-vision/utils/seed.py


In [6]:
%%writefile /content/conser-vision/src/data/dataset.py
"""
src/data/dataset.py
--------------------
PyTorch Dataset for camera-trap image classification.
"""
from __future__ import annotations

from pathlib import Path
from typing import Callable, Optional

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset


CLASS_NAMES = [
    "antelope_duiker", "bird", "blank", "civet_genet",
    "hog", "leopard", "monkey_prosimian", "rodent",
]
CLASS_TO_IDX: dict[str, int] = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS: dict[int, str] = {i: c for i, c in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)


class WildlifeDataset(Dataset):
    """Camera-trap image dataset for Conser-vision competition."""

    def __init__(
        self,
        df: pd.DataFrame,
        images_dir: str | Path,
        transform: Optional[Callable] = None,
        is_test: bool = False,
        id_col: str = "id",
    ) -> None:
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.transform = transform
        self.is_test = is_test
        self.id_col = id_col
        if not is_test:
            self.labels: np.ndarray = self.df[CLASS_NAMES].values.astype(np.float32)
        self.image_ids: list[str] = self.df[id_col].tolist()

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, idx: int) -> Image.Image:
        image_id = self.image_ids[idx]
        img_path = self.images_dir / f"{image_id}.jpg"
        if not img_path.exists():
            for ext in [".JPG", ".jpeg", ".JPEG", ".png", ".PNG"]:
                alt = self.images_dir / f"{image_id}{ext}"
                if alt.exists():
                    img_path = alt
                    break
        return Image.open(img_path).convert("RGB")

    def __getitem__(self, idx: int) -> dict:
        img = self._load_image(idx)
        if self.transform is not None:
            if type(self.transform).__module__.startswith("albumentations"):
                img_tensor: torch.Tensor = self.transform(image=np.array(img))["image"]
            else:
                img_tensor = self.transform(img)
        else:
            img_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        sample: dict = {"image": img_tensor, "id": self.image_ids[idx]}
        if not self.is_test:
            sample["label"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return sample


def load_dataframes(
    train_features_path: str,
    train_labels_path: str,
    test_features_path: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load and merge train/test DataFrames."""
    train_features = pd.read_csv(train_features_path)
    train_labels   = pd.read_csv(train_labels_path)
    test_df        = pd.read_csv(test_features_path)
    train_df = train_features.merge(train_labels, on="id", how="left")
    print(f"Train: {len(train_df):,} samples")
    print(f"Test:  {len(test_df):,} samples")
    print(f"Class distribution:\n{train_labels[CLASS_NAMES].sum().to_string()}")
    return train_df, test_df


Writing /content/conser-vision/src/data/dataset.py


In [7]:
%%writefile /content/conser-vision/src/data/transforms.py
"""
src/data/transforms.py
-----------------------
Augmentation pipelines using torchvision.transforms.v2.
"""
from __future__ import annotations

import torch
import torchvision.transforms.v2 as T
from torchvision.transforms.v2 import InterpolationMode

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def get_train_transforms(image_size: int = 224) -> T.Compose:
    """Strong augmentation for training (camera-trap specific)."""
    return T.Compose([
        T.RandomResizedCrop(image_size, scale=(0.6, 1.0), ratio=(0.75, 1.33),
                            interpolation=InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.2),
        T.RandomRotation(degrees=15, interpolation=InterpolationMode.BICUBIC),
        T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
        T.RandomGrayscale(p=0.1),
        T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))], p=0.2),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def get_val_transforms(image_size: int = 224) -> T.Compose:
    """Deterministic pipeline for validation and test."""
    resize_size = int(image_size * (256 / 224))
    return T.Compose([
        T.Resize(resize_size, interpolation=InterpolationMode.BICUBIC),
        T.CenterCrop(image_size),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def get_tta_transforms(image_size: int = 224) -> list[T.Compose]:
    """TTA: 5 fully deterministic transforms (no RandomCrop).

    FiveCrop order: top-left, top-right, bottom-left, bottom-right, center.
    One Compose per crop position — all deterministic and reproducible.
    """
    resize_size = int(image_size * (256 / 224))
    norm = [
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
    resize    = T.Resize(resize_size, interpolation=InterpolationMode.BICUBIC)
    five_crop = T.FiveCrop(image_size)

    return [
        T.Compose([resize, five_crop, T.Lambda(lambda crops, idx=i: crops[idx]), *norm])
        for i in range(5)
    ]



Writing /content/conser-vision/src/data/transforms.py


In [8]:
%%writefile /content/conser-vision/src/models/model.py
"""
src/models/model.py
--------------------
Image classifier built on top of timm pretrained backbones.
"""
from __future__ import annotations

import torch
import torch.nn as nn
from timm.models import create_model


class WildlifeClassifier(nn.Module):
    """Wildlife species classifier with a timm backbone."""

    def __init__(self, model_name: str = "efficientnet_b3", num_classes: int = 8,
                 pretrained: bool = True, dropout: float = 0.3) -> None:
        super().__init__()
        self.model_name  = model_name
        self.num_classes = num_classes
        self.backbone = create_model(model_name, pretrained=pretrained,
                                     num_classes=0, global_pool="avg")
        num_features: int = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(num_features),
            nn.Dropout(p=dropout),
            nn.Linear(num_features, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))

    def get_optimizer_param_groups(self, lr: float = 1e-4,
                                   lr_backbone_multiplier: float = 0.1) -> list[dict]:
        return [
            {"params": self.backbone.parameters(), "lr": lr * lr_backbone_multiplier, "name": "backbone"},
            {"params": self.head.parameters(),     "lr": lr,                          "name": "head"},
        ]


def build_model(model_name: str, num_classes: int = 8, pretrained: bool = True,
                dropout: float = 0.3, checkpoint_path: str | None = None) -> WildlifeClassifier:
    """Instantiate a WildlifeClassifier, optionally loading from checkpoint."""
    model = WildlifeClassifier(model_name=model_name, num_classes=num_classes,
                               pretrained=pretrained, dropout=dropout)
    if checkpoint_path is not None:
        state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        if "model_state_dict" in state:
            state = state["model_state_dict"]
        model.load_state_dict(state)
        print(f"Loaded checkpoint from {checkpoint_path}")
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Model: {model_name}  |  Params: {n_params:.1f}M  |  Classes: {num_classes}")
    return model


Writing /content/conser-vision/src/models/model.py


In [ ]:
%%writefile /content/conser-vision/src/training/train.py
"""
src/training/train.py
----------------------
Training loop with site-aware (StratifiedGroupKFold) cross-validation.
Supports mixed-precision, early stopping, and checkpointing.
"""

from __future__ import annotations

import random
import time
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import log_loss
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.data.dataset import WildlifeDataset, CLASS_NAMES, NUM_CLASSES
from src.data.transforms import get_train_transforms, get_val_transforms
from src.models.model import build_model
from utils.seed import set_global_seed


def mixup_data(
    x: torch.Tensor,
    y: torch.Tensor,
    alpha: float = 0.4,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    """Apply MixUp augmentation to a batch.

    Samples a mixing coefficient lam ~ Beta(alpha, alpha) and returns
    a linearly interpolated image and the two original label tensors.

    Args:
        x: Image batch tensor, shape (B, C, H, W).
        y: Label tensor (one-hot or class indices), shape (B, ...).
        alpha: Beta distribution concentration parameter.

    Returns:
        Tuple of (mixed_x, y_a, y_b, lam) where lam is the scalar mixing coefficient.
    """
    lam = float(np.random.beta(alpha, alpha))
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a = y
    y_b = y[index]
    return mixed_x, y_a, y_b, lam


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler: GradScaler,
    device: torch.device,
    gradient_clip: float = 1.0,
    mixup_alpha: float = 0.4,
    mixup_prob: float = 0.0,
) -> dict[str, float]:
    """Single training epoch."""
    model.train()
    total_loss = 0.0
    n_samples = 0

    for batch in tqdm(loader, desc="  Train", leave=False):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=scaler.is_enabled()):
            if random.random() < mixup_prob:
                images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=mixup_alpha)
                logits = model(images)
                loss = lam * criterion(logits, labels_a) + (1 - lam) * criterion(logits, labels_b)
            else:
                logits = model(images)
                loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        n_samples += images.size(0)

    return {"loss": total_loss / n_samples}


@torch.no_grad()
def validate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict[str, float]:
    """Validation epoch. Returns loss and OOF predictions."""
    model.eval()
    total_loss = 0.0
    n_samples = 0
    all_probs: list[np.ndarray] = []
    all_logits: list[np.ndarray] = []
    all_labels: list[np.ndarray] = []

    for batch in tqdm(loader, desc="  Val", leave=False):
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        with autocast("cuda", enabled=device.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, labels)

        raw_logits = logits.cpu().numpy().astype(np.float32)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_logits.append(raw_logits)
        all_probs.append(probs)
        all_labels.append(labels.cpu().numpy())

        total_loss += loss.item() * images.size(0)
        n_samples += images.size(0)

    all_probs_np = np.concatenate(all_probs)
    all_logits_np = np.concatenate(all_logits)
    all_labels_np = np.concatenate(all_labels)
    # Convert one-hot to class indices for log_loss
    true_classes = all_labels_np.argmax(axis=1)
    ll = log_loss(true_classes, all_probs_np, labels=list(range(NUM_CLASSES)))
    label_indices = all_labels_np.argmax(axis=1)  # shape (n,) integer class indices

    return {
        "loss": total_loss / n_samples,
        "log_loss": ll,
        "probs": all_probs_np,
        "logits": all_logits_np,
        "labels": all_labels_np,       # one-hot, shape (n, 8) — kept for backward compat
        "label_indices": label_indices, # integer class indices, shape (n,)
    }


def train_fold(
    fold: int,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    images_dir: str,
    cfg: dict,
    device: torch.device,
    output_dir: Path,
    resume_checkpoint: Optional[Path] = None,
) -> tuple[float, np.ndarray]:
    """Train a single fold and return (best_val_log_loss, oof_predictions)."""
    print(f"\n{'='*60}", flush=True)
    print(f"  FOLD {fold+1}", flush=True)
    print(f"{'='*60}", flush=True)

    set_global_seed(cfg["general"]["seed"] + fold)
    model_cfg = cfg["baseline"]

    # Datasets & loaders
    train_ds = WildlifeDataset(
        train_df, images_dir,
        transform=get_train_transforms(model_cfg["image_size"])
    )
    val_ds = WildlifeDataset(
        val_df, images_dir,
        transform=get_val_transforms(model_cfg["image_size"])
    )
    # num_workers=0 on CPU/Windows (multiprocessing overhead is worse than single-process)
    _num_workers = 0 if device.type != "cuda" else 2
    train_loader = DataLoader(
        train_ds, batch_size=model_cfg["batch_size"],
        shuffle=True, num_workers=_num_workers, pin_memory=device.type == "cuda"
    )
    val_loader = DataLoader(
        val_ds, batch_size=model_cfg["batch_size"] * 2,
        shuffle=False, num_workers=_num_workers, pin_memory=device.type == "cuda"
    )

    # Model
    model = build_model(
        model_name=model_cfg["model_name"],
        num_classes=cfg["general"]["num_classes"],
        pretrained=model_cfg["pretrained"],
        dropout=model_cfg["dropout"],
    ).to(device)

    # Optimizer with differential LR
    param_groups = model.get_optimizer_param_groups(
        lr=model_cfg["learning_rate"],
        lr_backbone_multiplier=0.1,
    )
    optimizer = AdamW(
        param_groups,
        weight_decay=model_cfg["weight_decay"],
    )
    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=model_cfg["num_epochs"],
        eta_min=1e-6,
    )

    # Loss: soft cross-entropy with label smoothing
    criterion = nn.CrossEntropyLoss(label_smoothing=model_cfg["label_smoothing"])
    scaler = GradScaler("cuda", enabled=model_cfg["mixed_precision"] and device.type == "cuda")

    best_log_loss = float("inf")
    best_oof: Optional[np.ndarray] = None
    patience_counter = 0
    patience = model_cfg["early_stopping_patience"]
    start_epoch = 0

    if resume_checkpoint is not None and resume_checkpoint.exists():
        print(f"  Resuming from {resume_checkpoint}", flush=True)
        ckpt = torch.load(resume_checkpoint, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        best_log_loss = ckpt["best_score"]
        patience_counter = ckpt.get("patience_counter", 0)
        print(
            f"  Resumed: starting at epoch {start_epoch + 1}, "
            f"best log-loss so far: {best_log_loss:.4f}, "
            f"patience: {patience_counter}/{patience}",
            flush=True,
        )
        # Recover best_oof from the best checkpoint so OOF predictions are
        # always from the best model, not from the (potentially worse) last state.
        best_ckpt_path = output_dir / f"fold{fold+1}_best.pth"
        print("  Running initial validation to recover OOF predictions...", flush=True)
        if best_ckpt_path.exists():
            best_ckpt = torch.load(best_ckpt_path, map_location=device)
            model.load_state_dict(best_ckpt["model_state_dict"])
            best_oof = validate(model, val_loader, criterion, device)["probs"]
            # Restore the last (training) model state for continued training
            model.load_state_dict(ckpt["model_state_dict"])
        else:
            best_oof = validate(model, val_loader, criterion, device)["probs"]
        if start_epoch >= model_cfg["num_epochs"]:
            print("  All epochs already completed — skipping fold.", flush=True)
            return best_log_loss, best_oof

    for epoch in range(start_epoch, model_cfg["num_epochs"]):
        t0 = time.time()
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler, device,
            gradient_clip=model_cfg["gradient_clip"],
            mixup_alpha=model_cfg.get("mixup_alpha", 0.4),
            mixup_prob=model_cfg.get("mixup_prob", 0.0),
        )
        val_metrics = validate(model, val_loader, criterion, device)
        scheduler.step()

        elapsed = time.time() - t0
        print(
            f"  Epoch {epoch+1:3d}/{model_cfg['num_epochs']} | "
            f"Train loss: {train_metrics['loss']:.4f} | "
            f"Val loss: {val_metrics['loss']:.4f} | "
            f"Val log-loss: {val_metrics['log_loss']:.4f} | "
            f"LR: {scheduler.get_last_lr()[0]:.2e} | "
            f"{elapsed:.0f}s",
            flush=True,
        )

        if val_metrics["log_loss"] < best_log_loss:
            best_log_loss = val_metrics["log_loss"]
            best_oof = val_metrics["probs"]
            patience_counter = 0
            checkpoint_path = output_dir / f"fold{fold+1}_best.pth"
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "best_score": best_log_loss,
                    "val_log_loss": best_log_loss,
                    "patience_counter": patience_counter,
                    "fold": fold,
                    "val_logits": val_metrics["logits"],
                    "val_labels": val_metrics["label_indices"],
                    "val_sites": int(val_df["site"].nunique()) if "site" in val_df.columns else -1,
                },
                checkpoint_path,
            )
            print(f"  ✓ Saved best checkpoint → {checkpoint_path}", flush=True)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping triggered at epoch {epoch+1}", flush=True)
                # Save last checkpoint before exiting so resume works
                torch.save(
                    {
                        "epoch": epoch,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "scheduler_state_dict": scheduler.state_dict(),
                        "scaler_state_dict": scaler.state_dict(),
                        "best_score": best_log_loss,
                        "patience_counter": patience_counter,
                        "fold": fold,
                    },
                    output_dir / f"fold{fold+1}_last.pth",
                )
                break

        # Save last checkpoint every epoch for resume support
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "best_score": best_log_loss,
                "patience_counter": patience_counter,
                "fold": fold,
            },
            output_dir / f"fold{fold+1}_last.pth",
        )

    print(f"\n  Fold {fold+1} best log-loss: {best_log_loss:.4f}", flush=True)
    return best_log_loss, best_oof


def run_cv(
    train_df: pd.DataFrame,
    images_dir: str,
    cfg: dict,
    device: torch.device,
    output_dir: Path,
    resume: bool = False,
) -> tuple[np.ndarray, list[float]]:
    """Run full StratifiedGroupKFold (site-aware) cross-validation.

    Returns:
        oof_preds: OOF predictions array of shape (n_train, num_classes).
        fold_scores: List of per-fold log-loss values.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    n_splits = cfg["cross_validation"]["n_splits"]

    # --- Site-aware CV split ---
    # The public test set uses camera-trap sites that are COMPLETELY DISJOINT
    # from the train set (0 overlap, verified in reports/diagnostic_audit.md).
    # Therefore the local val fold MUST also be site-disjoint from the train
    # fold, otherwise the local metric measures in-site memorisation instead of
    # out-of-site generalisation (which is what the leaderboard measures).
    #
    # We use StratifiedGroupKFold(n_splits=5) unconditionally:
    #   - n_splits == 1  → take only the first fold (~20% site-disjoint holdout)
    #   - n_splits == 5  → iterate all 5 folds
    if "site" not in train_df.columns:
        raise KeyError(
            "train_df must contain a 'site' column for site-aware CV. "
            "Check that load_dataframes preserved it."
        )

    # Stratify on the class with highest probability (one-hot) and group on site
    y = train_df[CLASS_NAMES].values.argmax(axis=1)
    groups = train_df["site"].values

    oof_preds = np.zeros((len(train_df), cfg["general"]["num_classes"]), dtype=np.float32)
    fold_scores: list[float] = []
    all_val_idx: np.ndarray = np.array([], dtype=np.intp)

    sgkf = StratifiedGroupKFold(
        n_splits=5,
        shuffle=cfg["cross_validation"]["shuffle"],
        random_state=cfg["general"]["seed"],
    )
    all_splits = list(sgkf.split(np.arange(len(train_df)), y, groups))

    if n_splits == 1:
        # Fast-iteration mode: single site-disjoint hold-out (first fold only)
        splits = [all_splits[0]]
    elif n_splits == 5:
        splits = all_splits
    else:
        raise ValueError(
            f"cross_validation.n_splits must be 1 or 5, got {n_splits}. "
            "Site-aware CV is built from a fixed 5-way StratifiedGroupKFold."
        )

    for fold, (train_idx, val_idx) in enumerate(splits):
        # --- Guardrail: assert no site leakage between train and val folds ---
        train_groups = set(groups[train_idx])
        val_groups = set(groups[val_idx])
        leaked = train_groups & val_groups
        assert len(leaked) == 0, (
            f"GROUP LEAKAGE in fold {fold}: "
            f"{len(leaked)} sites shared between train and val folds"
        )
        # Per-fold logging (sites + class distribution sanity check)
        val_class_counts = np.bincount(y[val_idx], minlength=cfg["general"]["num_classes"])
        val_class_pct = val_class_counts / max(len(val_idx), 1) * 100
        print(
            f"[fold {fold}] "
            f"train rows={len(train_idx):>6}  val rows={len(val_idx):>6} | "
            f"train sites={len(train_groups):>4}  val sites={len(val_groups):>4}  leaked=0",
            flush=True,
        )
        print(
            "           val class dist: "
            + ", ".join(f"{c}={p:.1f}%" for c, p in zip(CLASS_NAMES, val_class_pct)),
            flush=True,
        )

        fold_train_df = train_df.iloc[train_idx]
        fold_val_df = train_df.iloc[val_idx]

        resume_checkpoint: Optional[Path] = None
        if resume:
            last_ckpt = output_dir / f"fold{fold+1}_last.pth"
            if last_ckpt.exists():
                resume_checkpoint = last_ckpt
            else:
                print(f"  No resume checkpoint found for fold {fold+1}, starting from scratch.", flush=True)

        fold_log_loss, fold_oof = train_fold(
            fold=fold,
            train_df=fold_train_df,
            val_df=fold_val_df,
            images_dir=images_dir,
            cfg=cfg,
            device=device,
            output_dir=output_dir,
            resume_checkpoint=resume_checkpoint,
        )

        oof_preds[val_idx] = fold_oof
        fold_scores.append(fold_log_loss)
        all_val_idx = np.concatenate([all_val_idx, val_idx])

    overall_ll = log_loss(
        train_df[CLASS_NAMES].values.argmax(axis=1)[all_val_idx],
        oof_preds[all_val_idx],
        labels=list(range(cfg["general"]["num_classes"])),
    )
    print(f"\n{'='*60}", flush=True)
    print("  CV Results:", flush=True)
    for i, s in enumerate(fold_scores):
        print(f"    Fold {i+1}: {s:.4f}", flush=True)
    print(f"  Mean fold log-loss: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}", flush=True)
    print(f"  OOF log-loss:       {overall_ll:.4f}", flush=True)
    print(f"{'='*60}", flush=True)

    # Save OOF predictions
    oof_df = train_df[["id"]].copy()
    for i, cls in enumerate(CLASS_NAMES):
        oof_df[cls] = oof_preds[:, i]
    oof_path = Path(cfg["paths"]["oof_dir"]) / "oof_predictions.csv"
    oof_df.to_csv(oof_path, index=False)
    print(f"  OOF predictions saved → {oof_path}", flush=True)

    return oof_preds, fold_scores

In [ ]:
%%writefile /content/conser-vision/src/evaluation/predict.py
"""
src/evaluation/predict.py
--------------------------
Generate test set predictions using trained fold checkpoints.
Supports averaging across folds and optional TTA.
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.data.dataset import WildlifeDataset, CLASS_NAMES
from src.data.transforms import get_val_transforms, get_tta_transforms
from src.evaluation.eval import calibrate_temperature
from src.models.model import build_model


@torch.no_grad()
def predict_single_checkpoint(
    model: torch.nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> np.ndarray:
    """Get softmax probabilities from a single model checkpoint."""
    model.eval()
    all_probs: list[np.ndarray] = []

    for batch in tqdm(loader, desc="  Predicting", leave=False):
        images = batch["image"].to(device, non_blocking=True)
        logits = model(images)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)

    return np.concatenate(all_probs)


@torch.no_grad()
def predict_logits_single_checkpoint(
    model: torch.nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> np.ndarray:
    """Get raw logits (pre-softmax) from a single model checkpoint."""
    model.eval()
    all_logits: list[np.ndarray] = []

    for batch in tqdm(loader, desc="  Predicting logits", leave=False):
        images = batch["image"].to(device, non_blocking=True)
        logits = model(images)
        all_logits.append(logits.cpu().numpy().astype(np.float32))

    return np.concatenate(all_logits)


@torch.no_grad()
def predict_with_tta(
    model: torch.nn.Module,
    test_df: pd.DataFrame,
    images_dir: str,
    image_size: int,
    device: torch.device,
    batch_size: int = 32,
    temperature: float | None = None,
) -> np.ndarray:
    """Predict with Test Time Augmentation: average over multiple views.

    When temperature is provided, collects raw logits per TTA view, averages
    them in logit space, then applies softmax(avg_logits / temperature).
    Otherwise averages softmax probabilities directly (legacy behaviour, T=1.0).
    """
    tta_transforms = get_tta_transforms(image_size)

    _num_workers = 0 if (device.type != "cuda" or sys.platform == "win32") else 4
    _pin_memory = device.type == "cuda"

    if temperature is not None:
        # Collect logits per TTA view, average in logit space, then apply softmax with T
        tta_logits: list[np.ndarray] = []
        for i, transform in enumerate(tta_transforms):
            print(f"  TTA augmentation {i+1}/{len(tta_transforms)}")
            ds = WildlifeDataset(test_df, images_dir, transform=transform, is_test=True)
            loader = DataLoader(
                ds, batch_size=batch_size, shuffle=False,
                num_workers=_num_workers, pin_memory=_pin_memory,
            )
            logits = predict_logits_single_checkpoint(model, loader, device)
            tta_logits.append(logits)

        avg_logits = np.mean(tta_logits, axis=0)
        scaled = torch.from_numpy((avg_logits / temperature).astype(np.float32))
        return F.softmax(scaled, dim=1).numpy()
    else:
        # Legacy: average probabilities across TTA views
        tta_probs: list[np.ndarray] = []
        for i, transform in enumerate(tta_transforms):
            print(f"  TTA augmentation {i+1}/{len(tta_transforms)}")
            ds = WildlifeDataset(test_df, images_dir, transform=transform, is_test=True)
            loader = DataLoader(
                ds, batch_size=batch_size, shuffle=False,
                num_workers=_num_workers, pin_memory=_pin_memory,
            )
            probs = predict_single_checkpoint(model, loader, device)
            tta_probs.append(probs)

        return np.mean(tta_probs, axis=0)


def generate_submission(
    test_df: pd.DataFrame,
    images_dir: str,
    checkpoint_dir: str | Path,
    cfg: dict,
    device: torch.device,
    use_tta: bool = True,
    output_path: str | None = None,
) -> pd.DataFrame:
    """Generate final submission by averaging predictions across all fold checkpoints.

    For each checkpoint, loads saved val_logits/val_labels (if present) to calibrate
    temperature T via NLL minimisation. Applies softmax(logits / T) for test predictions.
    Falls back to T=1.0 with a warning for old checkpoints without calibration data.

    Args:
        test_df: Test features DataFrame.
        images_dir: Directory containing test images.
        checkpoint_dir: Directory containing fold*.pth checkpoints.
        cfg: Config dict.
        device: Torch device.
        use_tta: Whether to use TTA.
        output_path: If provided, saves the submission CSV here.

    Returns:
        Submission DataFrame with columns [id, antelope_duiker, bird, ..., rodent].
    """
    checkpoint_dir = Path(checkpoint_dir)
    checkpoints = sorted(checkpoint_dir.glob("fold*_best.pth"))
    print(f"Found {len(checkpoints)} fold checkpoints: {[c.name for c in checkpoints]}")
    if not checkpoints:
        raise FileNotFoundError(
            f"No fold checkpoints found in {checkpoint_dir}. "
            "Run training first."
        )

    model_cfg = cfg["baseline"]
    all_fold_probs: list[np.ndarray] = []

    for ckpt_path in checkpoints:
        print(f"\nLoading {ckpt_path.name}")

        # Load checkpoint dict to retrieve val_logits / val_labels for calibration
        ckpt_dict = torch.load(ckpt_path, map_location=device)
        if "val_logits" in ckpt_dict and "val_labels" in ckpt_dict:
            val_logits: np.ndarray = ckpt_dict["val_logits"]
            val_labels: np.ndarray = ckpt_dict["val_labels"]
            temperature = calibrate_temperature(val_logits, val_labels)
            # Report whether the fold's val set is site-disjoint. After the
            # CV fix (2026-04-10), these logits/labels come from the
            # StratifiedGroupKFold val fold, so calibration is site-disjoint
            # by construction. val_sites may be absent in legacy checkpoints.
            n_val = int(val_labels.shape[0])
            n_sites = int(ckpt_dict.get("val_sites", -1))
            sites_str = f"{n_sites} unique sites" if n_sites >= 0 else "N unique sites unknown (legacy checkpoint)"
            print(
                f"[calibration] T = {temperature:.4f} fit on val fold with "
                f"{n_val} samples, {sites_str}"
            )
        else:
            print(
                f"  WARNING: checkpoint {ckpt_path.name} has no val_logits/val_labels. "
                "Falling back to T=1.0 (no calibration)."
            )
            temperature = 1.0

        model = build_model(
            model_name=model_cfg["model_name"],
            num_classes=cfg["general"]["num_classes"],
            pretrained=False,
            dropout=model_cfg["dropout"],
            checkpoint_path=str(ckpt_path),
        ).to(device)

        if use_tta:
            probs = predict_with_tta(
                model, test_df, images_dir,
                image_size=model_cfg["image_size"],
                device=device,
                batch_size=model_cfg["batch_size"],
                temperature=temperature,
            )
        else:
            ds = WildlifeDataset(
                test_df, images_dir,
                transform=get_val_transforms(model_cfg["image_size"]),
                is_test=True,
            )
            _num_workers = 0 if (device.type != "cuda" or sys.platform == "win32") else 4
            loader = DataLoader(
                ds, batch_size=model_cfg["batch_size"] * 2, shuffle=False,
                num_workers=_num_workers, pin_memory=device.type == "cuda",
            )
            raw_logits = predict_logits_single_checkpoint(model, loader, device)
            scaled = torch.from_numpy((raw_logits / temperature).astype(np.float32))
            probs = F.softmax(scaled, dim=1).numpy()

        all_fold_probs.append(probs)
        del model
        torch.cuda.empty_cache()

    # Average across folds
    final_probs = np.mean(all_fold_probs, axis=0)

    # Build submission DataFrame
    sub_df = pd.DataFrame({"id": test_df["id"].values})
    for i, cls in enumerate(CLASS_NAMES):
        sub_df[cls] = final_probs[:, i]

    # Verify probabilities sum to ~1
    row_sums = sub_df[CLASS_NAMES].sum(axis=1)
    assert (row_sums - 1.0).abs().max() < 1e-4, "Probabilities don't sum to 1!"

    if output_path is not None:
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        sub_df.to_csv(output_path, index=False)
        print(f"\nSubmission saved -> {output_path}")
        print(f"  Shape: {sub_df.shape}")
        print(f"  Preview:\n{sub_df.head()}")

    return sub_df

In [ ]:
%%writefile /content/conser-vision/configs/config.yaml
# =============================================================================
# Config — Conser-vision: Wildlife Image Classification
# Competition: https://www.drivendata.org/competitions/87/
# Metric: Log-loss (multiclass, 8 classes)
# =============================================================================

general:
  seed: 42
  competition: "conser-vision"
  task: "multiclass_classification"
  metric: "log_loss"
  num_classes: 8
  class_names:
    - "antelope_duiker"
    - "bird"
    - "blank"
    - "civet_genet"
    - "hog"
    - "leopard"
    - "monkey_prosimian"
    - "rodent"
  n_jobs: -1

data:
  train_features_path: "data/raw/train_features.csv"
  train_labels_path: "data/raw/train_labels.csv"
  test_features_path: "data/raw/test_features.csv"
  train_images_dir: "data/raw/train_features"
  test_images_dir: "data/raw/test_features"
  id_col: "id"
  image_col: "filepath"        # colonna nel CSV con path relativo immagine
  image_size: 224
  image_size_large: 384        # per fase avanzata

cross_validation:
  # Site-aware CV: StratifiedGroupKFold(n_splits=5) on the `site` column.
  # n_splits=1 → use the first fold only (~20% site-disjoint hold-out, fast mode)
  # n_splits=5 → iterate all 5 folds (final ensemble mode)
  strategy: "StratifiedGroupKFold"
  group_col: "site"
  n_splits: 1
  shuffle: true

# ---- Baseline Model (EfficientNet-B3) ----
baseline:
  model_name: "tf_efficientnetv2_m"   # timm model name
  pretrained: true
  dropout: 0.3
  image_size: 224
  batch_size: 32
  num_epochs: 60
  learning_rate: 1.0e-4
  weight_decay: 1.0e-4
  label_smoothing: 0.1
  early_stopping_patience: 10
  mixed_precision: true
  gradient_clip: 1.0
  warmup_epochs: 2
  num_workers: 2
  mixup_alpha: 0.4
  mixup_prob: 0.5

# ---- Advanced Model (ConvNeXt-Base) ----
advanced:
  model_name: "convnext_base"
  pretrained: true
  dropout: 0.3
  image_size: 384
  batch_size: 16
  num_epochs: 40
  learning_rate: 5.0e-5
  weight_decay: 5.0e-5
  label_smoothing: 0.1
  early_stopping_patience: 7
  mixed_precision: true
  gradient_clip: 1.0
  warmup_epochs: 3
  num_workers: 2

# ---- Augmentation ----
augmentation:
  train:
    random_resized_crop: true
    horizontal_flip: 0.5
    vertical_flip: 0.3
    rotation_degrees: 15
    color_jitter:
      brightness: 0.3
      contrast: 0.3
      saturation: 0.2
      hue: 0.1
    random_grayscale: 0.05      # camera traps spesso B&W
    gaussian_blur: 0.2
    normalize:
      mean: [0.485, 0.456, 0.406]
      std: [0.229, 0.224, 0.225]
  val:
    resize: 256
    center_crop: 224
    normalize:
      mean: [0.485, 0.456, 0.406]
      std: [0.229, 0.224, 0.225]

# ---- TTA ----
tta:
  enabled: true
  n_augments: 5

# ---- Ensemble ----
ensemble:
  method: "average"   # average | weighted_average | rank_average
  weights: null

# ---- LR Scheduler ----
scheduler:
  name: "CosineAnnealingLR"   # CosineAnnealingLR | OneCycleLR | CosineAnnealingWarmRestarts
  T_max: 30                   # epochs
  eta_min: 1.0e-6

# ---- Paths ----
paths:
  model_dir: "models/weights"
  checkpoint_dir: "models/checkpoints"
  submission_dir: "submissions"
  oof_dir: "data/processed"

# ---- Logging ----
logging:
  use_wandb: false
  wandb_project: "drivendata-conser-vision"
  run_name: null
  log_every_n_steps: 50

## 3 · Data Setup

**Option A (recommended):** upload `train_features.zip` directly from your PC — fastest, bypasses Drive latency.  
**Option B:** use the zip already on Drive under `DRIVE_DATA/train_features.zip`.

Run the cell below — it will ask which option you want.


In [12]:
# ── Load data into Colab RAM ───────────────────────────────────
# Option A: upload zip directly from your PC (fastest)
# Option B: copy zip from Drive (no upload needed)
#
# Set USE_DRIVE_ZIP = True to use the zip already on Drive,
# or False to trigger a file upload dialog.
USE_DRIVE_ZIP = True  # ← change to False to upload from PC

import os
import shutil
import zipfile
from pathlib import Path

LOCAL_DATA = Path("/content/data_local")

if LOCAL_DATA.exists():
    print("✓ Data already in RAM — skipping.")
else:
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)

    # ── Step 1: get the zip ────────────────────────────────────────
    if USE_DRIVE_ZIP:
        zip_src = DRIVE_DATA / "train_features.zip"
        if not zip_src.exists():
            raise FileNotFoundError(f"Zip not found on Drive: {zip_src}")
        zip_local = Path("/content/train_features.zip")
        print(f"Copiando zip da Drive → /content/ ...")
        shutil.copy2(str(zip_src), str(zip_local))
        print("  ✓ Zip copiato")
    else:
        from google.colab import files
        print("Seleziona train_features.zip dal tuo PC:")
        uploaded = files.upload()
        zip_name = list(uploaded.keys())[0]
        zip_local = Path(f"/content/{zip_name}")

    # ── Step 2: extract images ─────────────────────────────────────
    print(f"Estraendo {zip_local.name} → {LOCAL_DATA} ...")
    with zipfile.ZipFile(str(zip_local), "r") as zf:
        zf.extractall(str(LOCAL_DATA))
    zip_local.unlink()  # free space
    print("  ✓ Immagini estratte")

    # ── Step 3: copy CSV files from Drive ─────────────────────────
    for csv_file in ["train_features.csv", "train_labels.csv", "test_features.csv"]:
        src = DRIVE_DATA / csv_file
        if not src.exists():
            raise FileNotFoundError(f"CSV non trovato su Drive: {src}")
        shutil.copy2(str(src), str(LOCAL_DATA / csv_file))
        print(f"  ✓ {csv_file}")

    # ── Step 4: verify image dir (handle nested zip structure) ─────
    img_dir = LOCAL_DATA / "train_features"
    if not img_dir.exists():
        nested = img_dir / "train_features"
        if nested.exists():
            nested.rename(img_dir)
            print("  ✓ Corretta struttura zip annidata")
        else:
            print(f"  ⚠ Cartella train_features non trovata in {LOCAL_DATA}")
            print(f"    Contenuto: {list(LOCAL_DATA.iterdir())}")

    n_imgs = len(list((LOCAL_DATA / 'train_features').glob('*.jpg')))
    print(f"\n✓ Data pronto in RAM: {n_imgs:,} immagini train")


Copiando zip da Drive → /content/ ...
  ✓ Zip copiato
Estraendo train_features.zip → /content/data_local ...
  ✓ Immagini estratte
  ✓ train_features.csv
  ✓ train_labels.csv
  ✓ test_features.csv

✓ Data pronto in RAM: 16,488 immagini train


In [13]:
# ── Patch config con path locali + verifica ────────────────────
import os
import sys
import yaml
import pandas as pd

os.chdir("/content/conser-vision")
sys.path.insert(0, "/content/conser-vision")

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["train_features_path"] = str(LOCAL_DATA / "train_features.csv")
cfg["data"]["train_labels_path"]   = str(LOCAL_DATA / "train_labels.csv")
cfg["data"]["test_features_path"]  = str(LOCAL_DATA / "test_features.csv")
cfg["data"]["train_images_dir"]    = str(LOCAL_DATA / "train_features")
cfg["data"]["test_images_dir"]     = str(LOCAL_DATA / "test_features")

with open("configs/config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

# Verifica
train_df = pd.read_csv(cfg["data"]["train_features_path"])
labels   = pd.read_csv(cfg["data"]["train_labels_path"])
test_df  = pd.read_csv(cfg["data"]["test_features_path"])
CLASS_NAMES = cfg["general"]["class_names"]

print(f"Train samples : {len(train_df):,}")
print(f"Test samples  : {len(test_df):,}")
print("\nClass distribution:")
print(labels[CLASS_NAMES].sum().to_string())

sample_id  = train_df["id"].iloc[0]
sample_img = LOCAL_DATA / "train_features" / f"{sample_id}.jpg"
print(f"\nSample image exists: {sample_img.exists()}  ({sample_img})")


Train samples : 16,488
Test samples  : 4,464

Class distribution:
antelope_duiker     2474.0
bird                1641.0
blank               2213.0
civet_genet         2423.0
hog                  978.0
leopard             2254.0
monkey_prosimian    2492.0
rodent              2013.0

Sample image exists: True  (/content/data_local/train_features/ZJ000000.jpg)


## 4 · Training


In [14]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  TRAINING PARAMETERS — edit here                            ║
# ╚══════════════════════════════════════════════════════════════╝

N_FOLDS    = 5     # 5 for full CV; use 1 for a quick smoke-test
N_EPOCHS   = 30    # max epochs per fold (early stopping may stop earlier)
BATCH_SIZE = 32    # Colab T4 (~16 GB) fits batch_size=32 at 224px

# DRIVE_CKPT is defined in Cell 3 (Mount Drive)
# ╚══════════════════════════════════════════════════════════════╝

import yaml
import sys
sys.path.insert(0, "/content/conser-vision")

with open("/content/conser-vision/configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["cross_validation"]["n_splits"] = N_FOLDS
cfg["baseline"]["num_epochs"]        = N_EPOCHS
cfg["baseline"]["batch_size"]        = BATCH_SIZE

print("Config summary:")
print(f"  Model     : {cfg['baseline']['model_name']}")
print(f"  Image size: {cfg['baseline']['image_size']} px")
print(f"  Folds     : {N_FOLDS}")
print(f"  Epochs    : {N_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  LR        : {cfg['baseline']['learning_rate']}")


Config summary:
  Model     : efficientnet_b3
  Image size: 224 px
  Folds     : 5
  Epochs    : 30
  Batch size: 32
  LR        : 0.0001


In [17]:
# ── torch.load weights_only fix — already baked into train.py above ──
# Nothing to patch: weights_only=False is set directly in all torch.load
# calls in src/training/train.py (Cell 11).
print("✓ torch.load already has weights_only=False in source — no patch needed")

✓ Patched torch.load in train.py


In [18]:
# ── Run cross-validation training ─────────────────────────────
import os
import sys
import torch
os.chdir("/content/conser-vision")
sys.path.insert(0, "/content/conser-vision")

from pathlib import Path
from utils.seed import set_global_seed
from src.data.dataset import load_dataframes
from src.training.train import run_cv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

set_global_seed(cfg["general"]["seed"])

train_df, test_df = load_dataframes(
    train_features_path=cfg["data"]["train_features_path"],
    train_labels_path=cfg["data"]["train_labels_path"],
    test_features_path=cfg["data"]["test_features_path"],
)

output_dir = Path(cfg["paths"]["model_dir"])

oof_preds, fold_scores = run_cv(
    train_df=train_df,
    images_dir=cfg["data"]["train_images_dir"],
    cfg=cfg,
    device=device,
    output_dir=output_dir,
    resume_dir=DRIVE_CKPT,
)

print("\n✓ Training complete.")
print(f"  Mean CV log-loss: {sum(fold_scores)/len(fold_scores):.4f}")


Device: cuda
Train: 16,488 samples
Test:  4,464 samples
Class distribution:
antelope_duiker     2474.0
bird                1641.0
blank               2213.0
civet_genet         2423.0
hog                  978.0
leopard             2254.0
monkey_prosimian    2492.0
rodent              2013.0

  FOLD 1
Model: efficientnet_b3  |  Params: 10.7M  |  Classes: 8


/content/conser-vision/src/training/train.py:179: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler(enabled=model_cfg["mixed_precision"] and device.type == "cuda")


UnpicklingError: Weights only load failed. In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
Please file an issue with the following so that we can make `weights_only=True` compatible with your use case: WeightsUnpickler error: 

Can only build Tensor, Parameter, OrderedDict or types allowlisted via `add_safe_globals`, but got <class 'numpy.dtypes.Float16DType'>

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.

In [ ]:
# ── Backup checkpoints to Drive ────────────────────────────────
import shutil
from pathlib import Path

DRIVE_CKPT.mkdir(parents=True, exist_ok=True)

ckpt_dir = Path("/content/conser-vision/models/weights")
for f in ckpt_dir.glob("*.pth"):
    dest = DRIVE_CKPT / f.name
    shutil.copy2(f, dest)
    print(f"  Copied {f.name} → {dest}")

oof_src = Path("/content/conser-vision/data/processed/oof_predictions.csv")
if oof_src.exists():
    shutil.copy2(oof_src, DRIVE_CKPT / "oof_predictions.csv")
    print("  Copied oof_predictions.csv")

print("\n✓ Checkpoints backed up to Drive")


## 5 · Generate Submission


In [ ]:
# ── Generate submission CSV ────────────────────────────────────
import os
import sys
import torch
os.chdir("/content/conser-vision")
sys.path.insert(0, "/content/conser-vision")

from datetime import datetime
from pathlib import Path
import yaml

from src.data.dataset import load_dataframes
from src.evaluation.predict import generate_submission

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

# ── Options ───────────────────────────────────────────────────
USE_TTA  = True            # False = faster but lower quality
CKPT_DIR = "models/weights"  # or str(DRIVE_CKPT) if restoring from Drive
# ──────────────────────────────────────────────────────────────

_, test_df = load_dataframes(
    train_features_path=cfg["data"]["train_features_path"],
    train_labels_path=cfg["data"]["train_labels_path"],
    test_features_path=cfg["data"]["test_features_path"],
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
tta_tag   = "TTA" if USE_TTA else "noTTA"
out_path  = f"submissions/submission_{tta_tag}_{timestamp}.csv"

sub_df = generate_submission(
    test_df=test_df,
    images_dir=cfg["data"]["test_images_dir"],
    checkpoint_dir=CKPT_DIR,
    cfg=cfg,
    device=device,
    use_tta=USE_TTA,
    output_path=out_path,
)


In [ ]:
# ── Copy submission to Drive + download link ───────────────────
import shutil
from pathlib import Path
from google.colab import files

DRIVE_SUB.mkdir(parents=True, exist_ok=True)

local_sub  = Path(out_path)
drive_dest = DRIVE_SUB / local_sub.name
shutil.copy2(local_sub, drive_dest)
print(f"✓ Saved to Drive: {drive_dest}")

print(f"\nStarting download of {local_sub.name} ...")
files.download(str(local_sub))


## 6 · Restore Checkpoints from Drive *(optional)*
Use this section if you want to **continue training** or **predict** in a new session
using checkpoints saved to Drive in a previous session.


In [ ]:
# ── Restore checkpoints from Drive → local ─────────────────────
import shutil
from pathlib import Path

# DRIVE_CKPT is defined in Cell 3 (Mount Drive)

local_ckpt = Path("/content/conser-vision/models/weights")
local_ckpt.mkdir(parents=True, exist_ok=True)

restored = 0
for f in DRIVE_CKPT.glob("fold*_best.pth"):
    dest = local_ckpt / f.name
    shutil.copy2(f, dest)
    print(f"  Restored {f.name}")
    restored += 1

print(f"\n✓ Restored {restored} checkpoint(s)")
